# Ablation: Swap Probability

Comparing different swap probabilities for negative generation:
- **Swap=0.0**: No swapping (only in-place negatives)
- **Swap=0.5**: Medium swap probability
- **Swap=1.0 (Default)**: Full swapping (baseline)

The swap probability controls how negatives are generated:
- Low swap: More in-place modifications (changing attributes/relations in-place)
- High swap: More swapping of components between different positions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
from pathlib import Path

# Import shared ablation utilities
from ablation_utils import (
    setup_plotting_style,
    load_all_ablation_models,
    load_all_models_all_metrics,
    make_latex_ablation_table,
    plot_ablation_line,
    plot_ablation_bars,
    compute_deltas,
    print_summary,
    METRICS, METRIC_DISPLAY, METRIC_COLORS
)

# Set up plotting style
setup_plotting_style()

In [3]:
# =============================================================================
# CONFIGURATION - Define ablation models
# =============================================================================

ABLATION_MODELS = {
    "Swap=0.0": {
        "csv_path": "../evaluation/ablations/27-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_lr0.2_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap0.0_ablation_swap0.0_inplace1.0.csv",
        "is_baseline": False,
        "description": "No swapping (in-place only)",
        "swap_prob": 0.0
    },
    "Swap=0.5": {
        "csv_path": "../evaluation/ablations/27-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_lr0.2_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap0.5_ablation_swap0.5_inplace1.0.csv",
        "is_baseline": False,
        "description": "Medium swap probability",
        "swap_prob": 0.5
    },
    "CS-CLIP (Swap=1.0)": {
        "csv_path": "../evaluation/exp_csv/19-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.0_inplace1.0_swap1.0_csclip-negclip-hard-new_cleaned.csv",
        "is_baseline": True,
        "description": "Full swapping (baseline)",
        "swap_prob": 1.0
    },
}

# Primary metric for comparison
PRIMARY_METRIC = "text_contrastive_accuracy"

# Checkpoint selection (use best or specific step)
CHECKPOINT_STEP = None  # None = use best checkpoint, or specify step like 5000

# Ablation metadata
ABLATION_NAME = "SWAP PROBABILITY ABLATION"
PARAM_KEY = "swap_prob"
PARAM_LABEL = 'Swap Probability'

print("Ablation: Swap Probability")
print("="*50)
for name, cfg in ABLATION_MODELS.items():
    baseline_mark = " [BASELINE]" if cfg["is_baseline"] else ""
    print(f"  {name}{baseline_mark}: {cfg['description']}")

Ablation: Swap Probability
  Swap=0.0: No swapping (in-place only)
  Swap=0.5: Medium swap probability
  CS-CLIP (Swap=1.0) [BASELINE]: Full swapping (baseline)


In [4]:
# =============================================================================
# LOAD DATA - Single Metric (Primary)
# =============================================================================

scores_df = load_all_ablation_models(ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP)
print(f"\nLoaded {len(scores_df)} models, {len(scores_df.columns)} datasets")

Loading Swap=0.0...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=20000)
Loading Swap=0.5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=20000)
Loading CS-CLIP (Swap=1.0)...
[apply_mappings] Dropped 16 original rows replaced by aliased metrics
  Loaded 65 datasets (step=15000)

Common datasets (65): ['VL_CheckList/obj_location', 'VG_Attribution', 'VL_CheckList/attr_material', 'VL_CheckList/attr_action', 'VL_CheckList/rel_spatial', 'VL_CheckList/rel_action', 'VL_CheckList/obj_size', 'VG_Relation', 'VL_CheckList/attr_color', 'VL_CheckList/attr_size', 'ColorFoil', 'VL_CheckList/attr_state', 'MMVP/Spatial', 'ControlledImages/B', 'SugarCrepe/replace_att', 'ControlledImages/COCO-One', 'SugarCrepe/replace_obj', 'SugarCrepe++/swap_object', 'VALSE/actions', 'SPEC/absolute_size', 'MMVP/Quantity', 'SPEC/relative_size', 'ColorSwap', 'VALSE/relations', 'MMVP/Structural Character', 'SPEC/existence',

In [5]:
# =============================================================================
# DISPLAY RAW SCORES TABLE
# =============================================================================

# Convert to percentage and display
scores_pct = scores_df * 100

# Add average column
scores_pct['Average'] = scores_pct.mean(axis=1)

print("\n" + "="*60)
print(f"ABLATION: SWAP PROBABILITY")
print(f"Metric: {PRIMARY_METRIC}")
print("="*60)
display(scores_pct.round(1).style.highlight_max(axis=0, color='lightgreen'))


ABLATION: SWAP PROBABILITY
Metric: text_contrastive_accuracy


,VL_CheckList/obj_location,VG_Attribution,VL_CheckList/attr_material,VL_CheckList/attr_action,VL_CheckList/rel_spatial,VL_CheckList/rel_action,VL_CheckList/obj_size,VG_Relation,VL_CheckList/attr_color,VL_CheckList/attr_size,ColorFoil,VL_CheckList/attr_state,MMVP/Spatial,ControlledImages/B,SugarCrepe/replace_att,ControlledImages/COCO-One,SugarCrepe/replace_obj,SugarCrepe++/swap_object,VALSE/actions,SPEC/absolute_size,MMVP/Quantity,SPEC/relative_size,ColorSwap,VALSE/relations,MMVP/Structural Character,SPEC/existence,ControlledImages/VG-One,SugarCrepe/swap_att,SugarCrepe++/replace_object,VALSE/noun phrases,MMVP/State,ControlledImages/COCO-Two,SugarCrepe/replace_rel,SugarCrepe++/swap_atribute,VALSE/coreference,SPEC/absolute_spatial,Flickr30k_Order,MMVP/Color,SugarCrepe++/replace_attribute,NegBench/msr_vtt_mcq_rephrased_llama,ControlledImages/VG-Two,COLA/multi_objects,COCO-CF,SugarCrepe++/replace_relation,BLA/co,VALSE/existence,SugarCrepe/swap_obj,MMVP/Text,MMVP/Camera Perspective,BLA/ap,VisMin,MMVP/Presence,NegBench/VOC2007_mcq_llama3.1_rephrased,COCO_Order,NegBench/COCO_val_mcq_llama3.1_rephrased,ControlledImages/A,SugarCrepe/add_obj,VALSE/counting,SPEC/relative_spatial,MMVP/Orientation,Winoground,SugarCrepe/add_att,BLA/rc,VALSE/plurals,SPEC/count,Average
Model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Swap=0.0,94.000000,63.700000,73.500000,79.300000,77.800000,81.600000,93.500000,82.900000,78.700000,69.900000,88.900000,65.900000,40.000000,33.600000,89.800000,42.900000,95.500000,46.900000,76.500000,39.700000,0.000000,31.900000,58.300000,67.100000,6.700000,69.100000,45.500000,74.200000,92.300000,93.800000,6.700000,52.300000,78.500000,57.200000,59.100000,11.900000,61.100000,6.700000,80.300000,32.700000,49.600000,46.200000,76.800000,65.100000,49.200000,87.500000,72.200000,0.000000,26.700000,52.700000,78.400000,20.000000,38.800000,62.300000,30.500000,28.600000,93.300000,68.400000,28.800000,6.700000,33.200000,85.400000,49.900000,66.300000,31.000000,56.100000
Swap=0.5,94.400000,67.700000,74.800000,79.100000,79.400000,83.000000,93.100000,84.500000,79.800000,68.000000,89.900000,65.400000,40.000000,35.500000,88.500000,47.400000,95.000000,53.100000,84.800000,39.200000,6.700000,32.200000,58.700000,69.300000,33.300000,69.200000,44.100000,76.300000,91.400000,94.500000,6.700000,49.100000,79.000000,58.300000,56.000000,12.600000,95.700000,6.700000,77.500000,31.000000,52.700000,39.500000,77.000000,64.200000,48.700000,84.600000,70.600000,0.000000,13.300000,51.500000,77.300000,13.300000,36.600000,94.100000,30.300000,29.400000,92.600000,68.400000,28.800000,20.000000,29.000000,81.400000,50.200000,67.600000,29.000000,57.600000
CS-CLIP (Swap=1.0),94.100000,69.400000,74.500000,78.100000,80.700000,83.000000,93.100000,86.300000,79.200000,64.300000,90.500000,65.200000,46.700000,34.600000,86.200000,48.000000,94.900000,52.200000,83.200000,38.800000,6.700000,32.700000,59.000000,70.100000,6.700000,68.700000,45.100000,74.500000,91.700000,93.700000,13.300000,50.900000,79.700000,56.500000,56.200000,12.200000,96.700000,13.300000,74.200000,29.200000,53.100000,41.000000,78.200000,62.400000,48.400000,83.400000,69.000000,13.300000,6.700000,52.700000,78.600000,0.000000,37.800000,95.300000,31.400000,29.100000,90.800000,66.500000,29.500000,13.300000,29.800000,80.800000,49.600000,70.500000,34.500000,57.200000


In [6]:
# =============================================================================
# LOAD ALL METRICS (Text, Image, Group Contrastive Accuracy)
# =============================================================================

# Load all models with all metrics
all_metrics_df = load_all_models_all_metrics(ABLATION_MODELS, METRICS, CHECKPOINT_STEP)

# Extract just the summary columns (I2T, T2I, Group)
summary_cols = [col for col in ['I2T', 'T2I', 'Group'] if col in all_metrics_df.columns]
summary_df = all_metrics_df[summary_cols].copy()

# Add overall average
summary_df['Average'] = summary_df.mean(axis=1)

print("\n" + "="*60)
print("ABLATION: SWAP PROBABILITY - ALL METRICS")
print("="*60)
display((summary_df * 100).round(1).style.highlight_max(axis=0, color='lightgreen'))

Loading Swap=0.0...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading Swap=0.5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading CS-CLIP (Swap=1.0)...
[apply_mappings] Dropped 16 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']

Common datasets across all models (19): ['BLA', 'COCO-CF', 'COCO_Order', 'COLA', 'ColorFoil', 'ColorSwap', 'ControlledImages', 'Flickr30k_Order', 'MMVP', 'NegBench', 'SPEC', 'SugarCrepe', 'SugarCrepe++', 'VALSE', 'VG_Attribution', 'VG_Relation', 'VL_CheckList', 'VisMin', 'Winoground']

ABLATION: SWAP PROBABILITY - ALL METRICS


,I2T,T2I,Group,Average
Model,,,,
Swap=0.0,59.600000,41.300000,25.100000,42.000000
Swap=0.5,63.000000,40.400000,24.100000,42.500000
CS-CLIP (Swap=1.0),63.400000,41.600000,25.400000,43.400000


In [7]:
# =============================================================================
# LATEX TABLE GENERATION
# =============================================================================

# Generate LaTeX table
latex_table = make_latex_ablation_table(
    summary_df,
    ABLATION_MODELS,
    caption="Swap probability ablation. Controls the balance between in-place modifications and component swapping for negative generation. I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \\textbf{bold}, baseline \\underline{underlined}.",
    label="tab:ablation_swap_prob",
)

print("="*60)
print("LATEX TABLE")
print("="*60)
print(latex_table)

LATEX TABLE
\begin{table}[t]
  \centering
  \small
  \caption{Swap probability ablation. Controls the balance between in-place modifications and component swapping for negative generation. I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \textbf{bold}, baseline \underline{underlined}.}
  \label{tab:ablation_swap_prob}
  \begin{tabular}{lcccc}
    \toprule
    Model & I2T & T2I & Group & Average \\
    \midrule
    Swap=0.0 & 59.6 & 41.3 & 25.1 & 42.0 \\
    Swap=0.5 & 63.0 & 40.4 & 24.1 & 42.5 \\
    CS-CLIP (Swap=1.0) & \textbf{\underline{63.4}} & \textbf{\underline{41.6}} & \textbf{\underline{25.4}} & \textbf{\underline{43.4}} \\
    \bottomrule
  \end{tabular}
\end{table}


In [ ]:
# =============================================================================
# VISUALIZATION: GROUPED BAR CHART (All Metrics)
# =============================================================================

fig, ax = plot_ablation_bars(
    summary_df,
    ABLATION_MODELS,
    title='Swap Probability Ablation',
    save_path='../paper_figures/ablation_swap_prob_bars.pdf'
)

In [10]:
# =============================================================================
# SUMMARY
# =============================================================================

print_summary(summary_df, ABLATION_MODELS, ABLATION_NAME, PARAM_KEY)


SUMMARY: SWAP PROBABILITY ABLATION

Baseline: CS-CLIP (Swap=1.0)

Average Performance:
    Swap=0.0: 42.0% (-1.45pp vs baseline) | swap_prob=0.0
    Swap=0.5: 42.5% (-0.96pp vs baseline) | swap_prob=0.5
  ★ CS-CLIP (Swap=1.0): 43.4% (+0.00pp vs baseline) | swap_prob=1.0

Key Findings:
  - Best: CS-CLIP (Swap=1.0) (43.4%)
  - Worst: Swap=0.0 (42.0%)
  - Gap: 1.5pp


In [11]:
# =============================================================================
# DATASET-WISE AND SUBSET-WISE TABLES (with ARO merging)
# =============================================================================

from ablation_utils import (
    load_all_models_per_dataset,
    load_all_models_per_subset,
    make_latex_dataset_table,
    get_datasets_and_subsets,
    display_all_tables,
    load_benchmark_config
)

# Load benchmark config for dataset merge rules (e.g., ARO)
bench_cfg = load_benchmark_config()

# Display all tables for the primary metric (I2T) with ARO merging
dataset_df, subset_df, datasets_subsets = display_all_tables(
    ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP, 
    show_latex=True, apply_merge=True, benchmark_config=bench_cfg
)


PER-DATASET RESULTS (I2T)
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
[apply_mappings] Dropped 16 original rows replaced by aliased metrics


,ARO,BLA,COCO-CF,COLA,ColorFoil,ColorSwap,ControlledImages,MMVP,NegBench,SPEC,SugarCrepe,SugarCrepe++,VALSE,VL_CheckList,VisMin,Winoground,Average
Model,,,,,,,,,,,,,,,,,
Swap=0.0,67.500000,50.600000,76.800000,46.200000,88.900000,58.300000,42.100000,12.600000,34.000000,35.400000,84.100000,68.400000,74.100000,79.400000,78.400000,33.200000,58.100000
Swap=0.5,85.500000,50.200000,77.000000,39.500000,89.900000,58.700000,43.000000,15.600000,32.600000,35.200000,83.300000,68.900000,75.000000,79.700000,77.300000,29.000000,58.800000
CS-CLIP (Swap=1.0),86.900000,50.200000,78.200000,41.000000,90.500000,59.000000,43.500000,13.300000,32.800000,36.100000,82.200000,67.400000,74.800000,79.200000,78.600000,29.800000,59.000000



LaTeX:
\begin{table}[t]
  \centering
  \scriptsize
  \caption{Per-dataset I2T accuracy.}
  \label{tab:ablation_datasets_i2t}
  \begin{adjustbox}{max width=\textwidth}
  \begin{tabular}{lccccccccccccccccc}
    \toprule
    Model & \rotatebox{60}{ARO} & \rotatebox{60}{BLA} & \rotatebox{60}{COCO-CF} & \rotatebox{60}{COLA} & \rotatebox{60}{ColorFoil} & \rotatebox{60}{ColorSwap} & \rotatebox{60}{ControlledImages} & \rotatebox{60}{MMVP} & \rotatebox{60}{NegBench} & \rotatebox{60}{SPEC} & \rotatebox{60}{SugarCrepe} & \rotatebox{60}{SugarCrepe++} & \rotatebox{60}{VALSE} & \rotatebox{60}{VL\_CheckList} & \rotatebox{60}{VisMin} & \rotatebox{60}{Winoground} & \rotatebox{60}{Avg} \\
    \midrule
    Swap=0.0 & 67.5 & \textbf{50.6} & 76.8 & \textbf{46.2} & 88.9 & 58.3 & 42.1 & 12.6 & \textbf{34.0} & 35.4 & \textbf{84.1} & 68.4 & 74.1 & 79.4 & 78.4 & \textbf{33.2} & 58.1 \\
    Swap=0.5 & 85.5 & 50.2 & 77.0 & 39.5 & 89.9 & 58.7 & 43.0 & \textbf{15.6} & 32.6 & 35.2 & 83.3 & \textbf{68.9} & \textbf{7

,VL_CheckList/obj_location,ARO/VG_Attribution,VL_CheckList/attr_material,VL_CheckList/attr_action,VL_CheckList/rel_spatial,VL_CheckList/rel_action,VL_CheckList/obj_size,ARO/VG_Relation,VL_CheckList/attr_color,VL_CheckList/attr_size,ColorFoil,VL_CheckList/attr_state,MMVP/Spatial,ControlledImages/B,SugarCrepe/replace_att,ControlledImages/COCO-One,SugarCrepe/replace_obj,SugarCrepe++/swap_object,VALSE/actions,SPEC/absolute_size,MMVP/Quantity,SPEC/relative_size,ColorSwap,VALSE/relations,MMVP/Structural Character,SPEC/existence,ControlledImages/VG-One,SugarCrepe/swap_att,SugarCrepe++/replace_object,VALSE/noun phrases,MMVP/State,ControlledImages/COCO-Two,SugarCrepe/replace_rel,SugarCrepe++/swap_atribute,VALSE/coreference,SPEC/absolute_spatial,ARO/Flickr30k_Order,MMVP/Color,SugarCrepe++/replace_attribute,NegBench/msr_vtt_mcq_rephrased_llama,ControlledImages/VG-Two,COLA/multi_objects,COCO-CF,SugarCrepe++/replace_relation,BLA/co,VALSE/existence,SugarCrepe/swap_obj,MMVP/Text,MMVP/Camera Perspective,BLA/ap,VisMin,MMVP/Presence,NegBench/VOC2007_mcq_llama3.1_rephrased,ARO/COCO_Order,NegBench/COCO_val_mcq_llama3.1_rephrased,ControlledImages/A,SugarCrepe/add_obj,VALSE/counting,SPEC/relative_spatial,MMVP/Orientation,Winoground,SugarCrepe/add_att,BLA/rc,VALSE/plurals,SPEC/count,Average
Model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Swap=0.0,94.000000,63.700000,73.500000,79.300000,77.800000,81.600000,93.500000,82.900000,78.700000,69.900000,88.900000,65.900000,40.000000,33.600000,89.800000,42.900000,95.500000,46.900000,76.500000,39.700000,0.000000,31.900000,58.300000,67.100000,6.700000,69.100000,45.500000,74.200000,92.300000,93.800000,6.700000,52.300000,78.500000,57.200000,59.100000,11.900000,61.100000,6.700000,80.300000,32.700000,49.600000,46.200000,76.800000,65.100000,49.200000,87.500000,72.200000,0.000000,26.700000,52.700000,78.400000,20.000000,38.800000,62.300000,30.500000,28.600000,93.300000,68.400000,28.800000,6.700000,33.200000,85.400000,49.900000,66.300000,31.000000,56.100000
Swap=0.5,94.400000,67.700000,74.800000,79.100000,79.400000,83.000000,93.100000,84.500000,79.800000,68.000000,89.900000,65.400000,40.000000,35.500000,88.500000,47.400000,95.000000,53.100000,84.800000,39.200000,6.700000,32.200000,58.700000,69.300000,33.300000,69.200000,44.100000,76.300000,91.400000,94.500000,6.700000,49.100000,79.000000,58.300000,56.000000,12.600000,95.700000,6.700000,77.500000,31.000000,52.700000,39.500000,77.000000,64.200000,48.700000,84.600000,70.600000,0.000000,13.300000,51.500000,77.300000,13.300000,36.600000,94.100000,30.300000,29.400000,92.600000,68.400000,28.800000,20.000000,29.000000,81.400000,50.200000,67.600000,29.000000,57.600000
CS-CLIP (Swap=1.0),94.100000,69.400000,74.500000,78.100000,80.700000,83.000000,93.100000,86.300000,79.200000,64.300000,90.500000,65.200000,46.700000,34.600000,86.200000,48.000000,94.900000,52.200000,83.200000,38.800000,6.700000,32.700000,59.000000,70.100000,6.700000,68.700000,45.100000,74.500000,91.700000,93.700000,13.300000,50.900000,79.700000,56.500000,56.200000,12.200000,96.700000,13.300000,74.200000,29.200000,53.100000,41.000000,78.200000,62.400000,48.400000,83.400000,69.000000,13.300000,6.700000,52.700000,78.600000,0.000000,37.800000,95.300000,31.400000,29.100000,90.800000,66.500000,29.500000,13.300000,29.800000,80.800000,49.600000,70.500000,34.500000,57.200000



LaTeX:
\begin{table}[t]
  \centering
  \scriptsize
  \caption{Per-subset I2T accuracy.}
  \label{tab:ablation_subsets_i2t}
  \begin{adjustbox}{max width=\textwidth}
  \begin{tabular}{lcccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc}
    \toprule
    Model & \rotatebox{60}{VL\_CheckList/obj\_location} & \rotatebox{60}{ARO/VG\_Attribution} & \rotatebox{60}{VL\_CheckList/attr\_material} & \rotatebox{60}{VL\_CheckList/attr\_action} & \rotatebox{60}{VL\_CheckList/rel\_spatial} & \rotatebox{60}{VL\_CheckList/rel\_action} & \rotatebox{60}{VL\_CheckList/obj\_size} & \rotatebox{60}{ARO/VG\_Relation} & \rotatebox{60}{VL\_CheckList/attr\_color} & \rotatebox{60}{VL\_CheckList/attr\_size} & \rotatebox{60}{ColorFoil} & \rotatebox{60}{VL\_CheckList/attr\_state} & \rotatebox{60}{MMVP/Spatial} & \rotatebox{60}{ControlledImages/B} & \rotatebox{60}{SugarCrepe/replace\_att} & \rotatebox{60}{ControlledImages/COCO-One} & \rotatebox{60}{SugarCrepe/replace\_obj} & \rotatebox{60}{SugarCrepe